## Quandary Python Interface (New)

Content:
1. How to optimize for control pulses that realize a **unitary gate**
2. How to **simulate** the fidelity of (optimized or prescribed) control pulses
3. How to evaluate pulses on a specific sample rate
4. How to get the propagator for given control parameters
5. How to optimize for a **state-to-state** transfer
6. How to use the open-system solver to model decay and dephasing

In [ ]:
from quandary.new import *
import numpy as np
%matplotlib inline
import logging
logging.basicConfig(level=logging.WARNING)

### 1. How to optimize for control pulses that realize a **unitary gate**

In [ ]:
# Example: Optimize for pulses that realize a SWAP02 gate (swap 0th and 2nd state of a 3-level qudit)

# Define the number of modelled energy levels per qubit
Ne = [3]   # essential levels per qubit (dimension of the target gate)
Ng = [1]   # guard levels per qubit

# Define the qubit frequencies [GHz]
freq01 = [4.10595]  # 0-1 transition frequency per qubit [GHz]
selfkerr = [0.2198]  # anharmonicity per qubit [GHz]

# Define the target gate (SWAP02: swaps 0th and 2nd energy level)
unitary = [[0,0,1],[0,1,0],[1,0,0]]

# Define the pulse duration [ns] and maximum control amplitude [GHz]
pulse_length = 200.0
maxctrl = [0.005]  # [GHz] (= 5 MHz)

# Setup physics
setup = setup_quandary(
    nessential=Ne,
    nguard=Ng,
    transition_frequency=freq01,
    selfkerr=selfkerr,
    control_amplitude_bounds=maxctrl,
    final_time=pulse_length,
    spline_knot_spacing=3.0,
    control_zero_boundary_condition=False,
    output_directory="./run_dir_howto",
)

# Other optimization settings
setup.rand_seed = 34525
setup.optim_tol_grad_abs = 1e-4
setup.optim_tol_final_cost = 1e-4
setup.usematfree = True

# Run optimization
print(setup) # print the user specified settings
results = optimize(setup, targetgate=unitary, randomize_initial_control=True, control_initialization_amplitude=0.001, quiet=False, dry_run=False)
print(results.config) # print the configuration used for the optimization
print(f"\nOptimized Fidelity = {1.0 - results.infidelity}")

# Store the optimized B-spline parameters
popt = results.pcof.copy()

# Plot results (for single-qubit systems)
plot_results_1osc(results)

### 2. How to **simulate** the fidelity of (optimized or prescribed) control pulses

In [ ]:
# Simulate the dynamics of previously optimized control pulses and evaluate the fidelity

# Option (a): Pass B-spline control coefficients directly
results_sim = simulate(setup, pcof=popt, quiet=True)
print(f"Fidelity (from pcof) = {1.0 - results_sim.infidelity}")

# Option (b): Pass time-domain control pulses p(t) and q(t) directly (auto-downsampled to B-splines)
results_sim = simulate(setup, pt0=results.pt, qt0=results.qt, quiet=True)
print(f"Fidelity (from pt0/qt0) = {1.0 - results_sim.infidelity}")

### 3. How to evaluate control pulses on a specific sample rate

In [ ]:
# Evaluate the optimized B-spline controls on a different time grid (e.g. for hardware AWG)

# Define a sample rate (desired number of points per ns)
samplerate = 64

# Evaluate the controls on this sample rate
results_eval = evaluate_controls(setup, pcof=popt, points_per_ns=samplerate, quiet=True)

plot_pulse(results_eval)

### 4. How to get the propagator for given control parameters

In [ ]:
# results.uT contains the propagated states at final time T, for each initial state.
# If the initial states are the full basis (default), this is the solution operator / propagator.
propagator_T = results.uT

# If guard levels were used, uT is rectangular (nlevels x nessential).
# Extract the essential-level square (almost unitary) propagator:
nstates_ess = np.prod(Ne)
propagator = propagator_T[:nstates_ess, :]
print(f"Propagator:\n{propagator}")

# Compute fidelity from propagator directly
fid = np.abs(np.trace(propagator.conj().T @ unitary))**2 / nstates_ess**2
print(f"\nFidelity from propagator = {fid}")

### 5. How to optimize for a **state-to-state** transfer

In [ ]:
# Example: Optimize a state-to-state transfer |0> -> (|0> + |1>) / sqrt(2)

Ne = [2]   # Number of essential states per qubit
Ng = [0]   # Number of extra guard levels

# Define the initial and target state (in essential level dimensions)
initialcondition = [1.0, 0.0]
targetstate = [1.0/np.sqrt(2), 1.0/np.sqrt(2)]

# Pulse length and max amplitude
T = 100.0
maxctrl = [0.005]  # [GHz] (= 5 MHz)

# Setup physics
setup_s2s = setup_quandary(
    nessential=Ne,
    nguard=Ng,
    transition_frequency=[4.10595],
    final_time=T,
    control_amplitude_bounds=maxctrl,
    initial_levels=[0],       # Ground state |0>
    output_directory="./run_dir_s2s",
)
setup_s2s.rand_seed = 1234
setup_s2s.optim_tol_infidelity = 1e-5

# Run optimization (propagates only ONE initial state, not the full basis)
results_s2s = optimize(setup_s2s, targetstate=targetstate,
                       randomize_initial_control=True,
                       control_initialization_amplitude=0.0001,
                       quiet=False)
print(f"\nFidelity = {1.0 - results_s2s.infidelity}")

plot_results_1osc(results_s2s)

# The final state is in results.uT (one column, since only one initial condition)
realized_state = results_s2s.uT[:, 0]

# To get the full propagator, re-simulate with basis initial conditions
setup_basis = setup_s2s.copy()
setup_basis.initial_condition.condition_type = InitialConditionType.BASIS
results_basis = simulate(setup_basis, pcof=results_s2s.pcof, quiet=True)
propagator = results_basis.uT

# Verify: propagator applied to |0> should give the target state
test_fid = np.abs(np.array(targetstate).conj() @ propagator @ initialcondition)
print(f"Test fidelity = {test_fid}")

### 6. How to use the open-system solver to model decay and dephasing

In [ ]:
# Simulate dynamics under Lindbladian noise operators (decay and dephasing).
# The Lindblad solver is more expensive, so pre-optimize on the closed system first.

# State-to-state transfer configuration
Ne = [2]   # Number of essential states
Ng = [1]
targetstate = [1.0/np.sqrt(2), 1.0/np.sqrt(2)]
initialstate = [1.0, 0.0]     # Initial ground state
T = 100.0                      # Pulse length [ns]
maxctrl = [0.004]              # [GHz] (= 4 MHz)

T1 = [100000.0]  # T1 decay time [ns]
T2 = [80000.0]   # T2 dephasing time [ns]

# Setup with decoherence
setup_lblad = setup_quandary(
    nessential=Ne,
    nguard=Ng,
    transition_frequency=[4.10595],
    selfkerr=[0.2198],
    final_time=T,
    control_amplitude_bounds=maxctrl,
    initial_levels=[0],
    output_directory="./run_dir_open",
)
setup_lblad.decoherence_type = DecoherenceType.BOTH
setup_lblad.decay_time = T1
setup_lblad.dephase_time = T2
setup_lblad.optim_maxiter = 50

# For Lindblad solver, target state must be a density matrix rho = |psi><psi|
rho_target = np.outer(targetstate, np.conj(targetstate))

# Step 1: Pre-optimize on the closed system (no noise)
setup_closed = setup_lblad.copy()
setup_closed.decoherence_type = DecoherenceType.NONE
results_closed = optimize(setup_closed, targetstate=targetstate, quiet=False)
print(f"\nClosed-system fidelity = {1.0 - results_closed.infidelity}")

# Step 2: Evaluate closed-system pulses under noise
results_noisy = simulate(setup_lblad, pcof=results_closed.pcof, quiet=False)
print(f"\nFidelity under noise = {1.0 - results_noisy.infidelity}")

# Step 3: Re-optimize on the Lindblad solver using closed-system pulses as initial guess
results_reopt = optimize(setup_lblad, targetstate=rho_target, pcof=results_closed.pcof, quiet=False)
print(f"\nRe-optimized fidelity under noise = {1.0 - results_reopt.infidelity}")

plot_results_1osc(results_reopt)